# PCAP

Extração de **28 features globais de IPv4 em janelas de 1 segundo**, com referência de rótulos dos CSVs originais do CICFlowMeter-V3.

Este teste processa **até os primeiros 1.000.000 de registros PCAP**, incluindo registros não IPv4 na contagem do limite. As features e a cobertura dos rótulos consideram somente IPv4. Pacotes repetidos são preservados. O teste pode terminar antes do limite se atingir o corte temporal ou o fim dos arquivos.

O fluxo é: configurar → inspecionar entradas → preparar índice dos CSVs → extrair features e contagens em uma passagem → aplicar rótulo por janela → conferir resultados. As features não usam os rótulos. A rotulagem é exclusivamente uma referência offline para avaliação.

Extraia este pacote na raiz do projeto. O notebook passa a se chamar `PCAP.ipynb`; o arquivo antigo `PcapStatistics.ipynb`, se ainda existir, não é atualizado nem deve ser usado para este teste. Reinicie o kernel após substituir os módulos. `Flows.py` antigo pode permanecer no diretório, mas não é importado por este pipeline.


## 1. Dependências e configuração

Execute a instalação uma vez, se necessário. Ajuste **CSV_DIR** para a pasta contendo os CSVs originais completos do dia; não aponte para arquivos janelados, normalizados ou cenários. A busca de PCAPs e CSVs é recursiva. Nenhum arquivo de ataque é excluído automaticamente da referência: linhas benignas de qualquer arquivo podem ser necessárias.

**Fuso:** os PCAPs têm tempo epoch e a saída usa UTC. O CSV de exemplo tem horário sem fuso. `CSV_UTC_OFFSET_HOURS = 0` é uma hipótese inicial explícita para este teste, não uma correspondência já verificada. Se o CSV estiver em UTC−3, use `-3`; se estiver em UTC, use `0`. Não use o fuso do seu computador por conveniência. O corte também é interpretado nesse mesmo fuso. Não há ajuste automático de horário para aumentar a cobertura.

`CUTOFF_LOCAL` segue a data observada no CSV de exemplo, **2018-12-01**, e exclui pacotes a partir de 13:35:00. Confira a data nos seus arquivos. O encerramento antecipado pressupõe que cada arquivo esteja internamente ordenado; o teste verifica regressões apenas no prefixo efetivamente lido. Não encontrar inversões até o arquivo 35 não certifica os arquivos seguintes.

**Saída esperada:** configuração, quantidade de features (28), corte convertido para UTC e pasta do teste. Nenhuma extração ocorre nesta célula. As dependências podem imprimir avisos de captura ao vivo; esta rotina lê arquivos offline.


In [1]:
# Execute uma vez, se necessário:
# %pip install -r src/PcapStatistics/requirements.txt

from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "PcapStatistics").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Execute PCAP.ipynb na raiz do projeto ou em uma subpasta.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.PcapStatistics import (
    FEATURE_NAMES, PcapStatisticsDataset, FlowLabelIndex,
    discover_pcaps, discover_csvs, inspect_inputs, inspect_csvs,
    local_timestamp_ns, utc_timestamp, export_labeled_dataset,
)

PCAP_DIR = Path(r"E:\CICDDoS2019\01-12\PCAP")
CSV_DIR = Path(r"E:\CICDDoS2019\01-12\CSV")  # Ajuste para sua pasta de CSVs ORIGINAIS.
OUTPUT_DIR = PROJECT_ROOT / "output" / "PcapStatistics" / "test_1million"

WINDOW_SECONDS = 1.0
MAX_PACKETS = 1_000_000
PROGRESS_EVERY = 100_000
CSV_UTC_OFFSET_HOURS = 0  # HIPÓTESE inicial: conferir alinhamento PCAP/CSV.
CUTOFF_LOCAL = "2018-12-01 13:35:00"
KEEP_EMPTY_WINDOWS = True
REBUILD_LABEL_INDEX = False
CSV_CHUNK_SIZE = 100_000

CUTOFF_NS = local_timestamp_ns(CUTOFF_LOCAL, CSV_UTC_OFFSET_HOURS)
print("Features:", len(FEATURE_NAMES))
print("Limite de registros PCAP:", f"{MAX_PACKETS:,}")
print("Corte exclusivo em UTC:", utc_timestamp(CUTOFF_NS))
print("Offset CSV configurado:", CSV_UTC_OFFSET_HOURS, "horas — validar com os PCAPs")
print("Pasta de saída:", OUTPUT_DIR)


Features: 28
Limite de registros PCAP: 1,000,000
Corte exclusivo em UTC: 2018-12-01T13:35:00.000000000Z
Offset CSV configurado: 0 horas — validar com os PCAPs
Pasta de saída: c:\GitHub\anomaly-detection-data-stream\output\PcapStatistics\test_1million


## 2. Inspeção das entradas

Os PCAPs são reconhecidos pelo cabeçalho binário, inclusive os arquivos sem extensão. A inspeção lê **somente o primeiro registro de cada captura**. Na extração, os arquivos serão intercalados por timestamp, preservando sobreposições e mantendo uma antecipação por arquivo. Essa antecipação não integra o limite de um milhão de observações processadas.

Dos CSVs, esta inspeção lê somente três linhas por arquivo e verifica os oito campos necessários. `SampleLabels` é uma amostra, não a distribuição completa. O nome do arquivo não determina a classe: o exemplo `UDPLag.csv` contém `UDP-lag`, `WebDDoS` e `BENIGN`.

**Saída esperada:** quantidade de arquivos, tabela com primeiro timestamp UTC e tamanho de cada PCAP, e tabela de amostras dos CSVs. Esta célula não certifica a ordem integral nem a correspondência entre capturas e fluxos. Um caminho incorreto ou CSV sem os campos exigidos gera erro explícito.


In [2]:
pcapFiles = discover_pcaps(PCAP_DIR)
csvFiles = discover_csvs(CSV_DIR)
pcapRecords = inspect_inputs(pcapFiles)
pcapInventory = pd.DataFrame(pcapRecords)
csvInventory = pd.DataFrame(inspect_csvs(csvFiles))
print(f"PCAPs: {len(pcapFiles)} | CSVs: {len(csvFiles)}")
display(pcapInventory.drop(columns="FirstTimestampNs"))
display(csvInventory)

firstTimes = [row["FirstTimestampNs"] for row in pcapRecords if row["FirstTimestampNs"] is not None]
if not firstTimes:
    raise ValueError("As capturas não contêm registros.")
FIRST_PCAP_NS = min(firstTimes)
if FIRST_PCAP_NS >= CUTOFF_NS:
    raise ValueError("O primeiro pacote já está após o corte. Confira data e fuso.")
print("Primeiro pacote UTC:", utc_timestamp(FIRST_PCAP_NS))


PCAPs: 819 | CSVs: 11


,File,Bytes,FirstTimestamp
0,E:\CICDDoS2019\01-12\PCAP\SAT-01-12-2018_0,200000255,2018-12-01T13:17:10.711517000Z
1,E:\CICDDoS2019\01-12\PCAP\SAT-01-12-2018_01,200000146,2018-12-01T14:36:06.133220000Z
2,E:\CICDDoS2019\01-12\PCAP\SAT-01-12-2018_010,200000036,2018-12-01T14:36:23.516940000Z
3,E:\CICDDoS2019\01-12\PCAP\SAT-01-12-2018_0100,200000448,2018-12-01T14:40:25.389583000Z
4,E:\CICDDoS2019\01-12\PCAP\SAT-01-12-2018_0101,200000434,2018-12-01T14:40:28.090803000Z
...,...,...,...
814,E:\CICDDoS2019\01-12\PCAP\SAT-01-12-2018_095,200000112,2018-12-01T14:40:12.573412000Z
815,E:\CICDDoS2019\01-12\PCAP\SAT-01-12-2018_096,200000221,2018-12-01T14:40:15.490853000Z
816,E:\CICDDoS2019\01-12\PCAP\SAT-01-12-2018_097,200000116,2018-12-01T14:40:17.700108000Z
817,E:\CICDDoS2019\01-12\PCAP\SAT-01-12-2018_098,200000148,2018-12-01T14:40:20.174378000Z


,File,SampleTimestamp,SampleLabels,Bytes
0,E:\CICDDoS2019\01-12\CSV\DrDoS_DNS.csv,2018-12-01 10:51:39.813448,DrDoS_DNS,2133307948
1,E:\CICDDoS2019\01-12\CSV\DrDoS_LDAP.csv,2018-12-01 11:22:40.254769,DrDoS_LDAP,917302277
2,E:\CICDDoS2019\01-12\CSV\DrDoS_MSSQL.csv,2018-12-01 11:32:32.915441,DrDoS_MSSQL,1889181756
3,E:\CICDDoS2019\01-12\CSV\DrDoS_NetBIOS.csv,2018-12-01 11:47:08.463789,DrDoS_NetBIOS,1697479829
4,E:\CICDDoS2019\01-12\CSV\DrDoS_NTP.csv,2018-12-01 09:17:11.183810,"DrDoS_NTP, BENIGN",645008606
5,E:\CICDDoS2019\01-12\CSV\DrDoS_SNMP.csv,2018-12-01 12:00:13.902782,DrDoS_SNMP,2172579815
6,E:\CICDDoS2019\01-12\CSV\DrDoS_SSDP.csv,2018-12-01 12:23:13.663425,DrDoS_SSDP,1252686219
7,E:\CICDDoS2019\01-12\CSV\DrDoS_UDP.csv,2018-12-01 12:36:57.628026,DrDoS_UDP,1506039371
8,E:\CICDDoS2019\01-12\CSV\Syn.csv,2018-12-01 13:30:30.741451,Syn,637312127
9,E:\CICDDoS2019\01-12\CSV\TFTP.csv,2018-12-01 13:34:27.403713,TFTP,9302011115


Primeiro pacote UTC: 2018-12-01T13:17:10.711517000Z


## 3. Índice dos fluxos rotulados

Lemos em blocos os CSVs originais, usando somente IPs, portas, protocolo, início, duração e rótulo. O fim do fluxo é `Timestamp + Flow Duration` em microssegundos. Preservamos fluxos que começaram antes da captura, mas ainda podem sobrepor o período. O índice usa SQLite em disco; durante a extração, uma varredura temporal mantém em memória os intervalos ativos. Não há consulta SQL ou leitura de CSV para cada pacote.

Nesta versão a associação é conservadora para **TCP e UDP não fragmentados com cabeçalho disponível**. Fluxos com outros protocolos são contabilizados como não suportados. Seus pacotes IPv4 continuam nas features e ficam sem associação. Os intervalos são fechados `[início, fim]`, com tolerância temporal zero; fluxos de duração zero só podem corresponder ao instante exato.

A chave aceita ambos os sentidos da comunicação. Intervalos sobrepostos com o mesmo rótulo permitem associação de classe; rótulos diferentes geram ambiguidade. Referências repetidas não multiplicam nem removem pacotes. Isso herda o rótulo do fluxo, sem provar individualmente que cada pacote é malicioso.

O limite de um milhão vale para **PCAP**, não para a leitura de CSVs. A primeira preparação do índice pode demorar: todos os CSVs escolhidos são examinados. Nas execuções seguintes, o índice é reutilizado quando caminhos, tamanhos, datas de modificação e parâmetros coincidem. Para reconstruir explicitamente, use `REBUILD_LABEL_INDEX=True`.

**Saída esperada:** progresso por CSV, resumo de linhas lidas, fluxos indexados no período e distribuição das classes dos fluxos. Essas contagens são de **fluxos**, não de pacotes ou janelas. Se nada for indexado, confira os arquivos, data e fuso antes de executar a extração.


In [3]:
labelIndex = FlowLabelIndex.build(
    csv_paths=csvFiles,
    output_path=OUTPUT_DIR / "flow_label_index.sqlite",
    csv_utc_offset_hours=CSV_UTC_OFFSET_HOURS,
    start_ns=FIRST_PCAP_NS,
    cutoff_ns=CUTOFF_NS,
    chunksize=CSV_CHUNK_SIZE,
    rebuild=REBUILD_LABEL_INDEX,
)
indexSummary = {k: v for k, v in labelIndex.summary.items() if k not in ("Signature", "FlowLabels")}
display(pd.DataFrame([indexSummary]))
display(pd.Series(labelIndex.summary["FlowLabels"], name="Indexed_Flows").rename_axis("Label").to_frame())
if labelIndex.summary["IndexedFlows"] == 0:
    raise ValueError("Nenhum fluxo TCP/UDP no intervalo configurado. Confira data/fuso e os CSVs.")


CSV indexado: DrDoS_DNS.csv | linhas lidas acumuladas: 5,074,413
CSV indexado: DrDoS_LDAP.csv | linhas lidas acumuladas: 7,255,955
CSV indexado: DrDoS_MSSQL.csv | linhas lidas acumuladas: 11,780,453
CSV indexado: DrDoS_NetBIOS.csv | linhas lidas acumuladas: 15,875,439
CSV indexado: DrDoS_NTP.csv | linhas lidas acumuladas: 17,092,446
CSV indexado: DrDoS_SNMP.csv | linhas lidas acumuladas: 22,253,823
CSV indexado: DrDoS_SSDP.csv | linhas lidas acumuladas: 24,865,197
CSV indexado: DrDoS_UDP.csv | linhas lidas acumuladas: 28,001,999
CSV indexado: Syn.csv | linhas lidas acumuladas: 29,584,680
CSV indexado: TFTP.csv | linhas lidas acumuladas: 49,692,507
CSV indexado: UDPLag.csv | linhas lidas acumuladas: 50,063,112


,CsvFiles,RowsRead,IndexedFlows,UnsupportedProtocolFlows,CsvUtcOffsetHours,FirstFlowUTC,LastFlowEndUTC
0,11,50063112,2052507,40,0,2018-12-01T13:15:31.052421000Z,2018-12-01T13:36:37.765275000Z


,Indexed_Flows
Label,
Syn,1582286
BENIGN,2328
TFTP,149958
WebDDoS,257
UDP-lag,317678


## 4. Features e convenções

Cada linha descreve **todo o IPv4 observado em um segundo**, sem separar por host. Comprimento é o total declarado no cabeçalho IPv4, incluindo o cabeçalho IP, sem Ethernet. Capturas truncadas são contadas no resumo; seus tamanhos declarados continuam utilizáveis. Fragmentos entram nas estatísticas IP, mas não têm portas, flags ou rótulo inferidos sem remontagem.

| Grupo | Quantidade | Definição |
|---|---:|---|
| Volume | 2 | Contagem de IPv4 e soma dos bytes IP. Com 1s, equivalem numericamente a pacotes/s e bytes/s. |
| Tamanho | 4 | Mínimo, máximo, média e desvio populacional do comprimento IP. |
| IAT | 4 | Intervalos em microssegundos entre IPv4 consecutivos dentro da janela, mesmo de conexões diferentes. |
| Protocolos | 3 | Frações TCP, UDP e ICMP entre todos os pacotes IPv4. |
| Flags | 5 | Frações SYN, ACK, RST, FIN e SYN sem ACK entre todos os pacotes TCP IPv4. Cabeçalhos não disponíveis não geram flags. |
| Diversidade | 4 | IPs e portas distintos de origem e destino. Portas somente quando disponíveis. |
| Entropias | 4 | Shannon em bits, calculada sobre as frequências por pacote. Portas usam somente ocorrências disponíveis. |
| TTL | 2 | Média e desvio populacional do TTL IPv4. |

Médias e desvios usam atualização incremental. Entropias usam contadores por janela, cujo custo de memória cresce com a diversidade. Valores sem observações são zero nesta representação; uma janela sem IPv4 será identificada como `EMPTY`. IATs não atravessam janelas. Não há imputação aprendida, normalização, seleção de features ou reconstrução de conexões.

**Saída esperada:** tabela com exatamente os 28 nomes na ordem de exportação. Timestamp, completude e rótulos são metadados, não features.


In [4]:
display(pd.DataFrame({"Feature": FEATURE_NAMES}, index=range(1, len(FEATURE_NAMES) + 1)))


,Feature
1,Packet_Count
2,IP_Bytes_Total
3,Packet_Length_Min
4,Packet_Length_Max
5,Packet_Length_Mean
6,Packet_Length_Std
7,IAT_Min
8,IAT_Max
9,IAT_Mean
10,IAT_Std


## 5. Teste completo com o primeiro milhão de pacotes

Uma única passagem atualiza as features e associa os pacotes aos fluxos. Os rótulos não alteram os acumuladores de features. Ao terminar cada janela, gravamos `window_features.csv` e `window_label_counts.csv`. O primeiro contém as 28 features; o segundo preserva contagens de benigno, ataque, desconhecidos e de cada classe de ataque.

As janelas são alinhadas ao epoch UTC, no formato `[WindowStart, Timestamp)`. `Timestamp` é o término nominal exclusivo. A troca de PCAP não fecha uma janela. Lacunas entre registros geram janelas vazias quando `KEEP_EMPTY_WINDOWS=True`; uma lacuna não comprova silêncio real do sensor.

`WindowComplete=0` identifica a primeira janela quando a leitura começa depois de seu início, e sempre a última janela de maneira conservadora. Parar no milionésimo pacote pode cortar um segundo ao meio. A janela parcial permanece para inspeção e pode ser excluída posteriormente da avaliação. `WindowComplete=1` significa cobertura do intervalo de processamento, não garantia de ausência de perdas na captura.

**Saída esperada:** progresso a cada 100.000 registros, taxa observada de processamento e cobertura dos rótulos IPv4. Ao final, resumo com quantidade efetiva de pacotes e janelas, timestamps extremos, desconhecidos, ambiguidades, fragmentos e truncamentos. A taxa inclui a associação offline e a escrita; não é uma medição isolada da latência do detector real. O número de janelas depende da duração temporal dos pacotes, não é fixo.

Esta célula grava resultados em `test_1million` e substitui intermediários anteriores dessa pasta ao concluir. Execuções interrompidas não são retomadas automaticamente. A saída final é atualizada somente na próxima seção; execute ambas em sequência.


In [5]:
extractor = PcapStatisticsDataset(
    windowSeconds=WINDOW_SECONDS,
    maxPackets=MAX_PACKETS,
    keepEmptyWindows=KEEP_EMPTY_WINDOWS,
    progressEvery=PROGRESS_EVERY,
)
result = extractor.runFiles(
    pcapFiles=pcapFiles,
    outputDirectory=OUTPUT_DIR,
    labelIndex=labelIndex,
    cutoffNs=CUTOFF_NS,
)
display(pd.DataFrame([result.as_dict()]))
if result.labelCoverage == 0:
    print("ATENÇÃO: nenhuma associação de rótulo. Revise data, fuso e cobertura dos CSVs; não use para avaliar o modelo.")


100,000 pacotes | 889 janelas | 3,328 pacotes/s | cobertura IPv4: 0.04%


,featuresPath,countsPath,packetsProcessed,ipv4Packets,windowsWritten,packetLimitReached,elapsedSeconds,packetsPerSecond,labelCoverage,matchedPackets,unknownPackets,ambiguousPackets,truncatedPackets,fragmentedPackets,firstTimestamp,lastTimestamp
0,C:\GitHub\anomaly-detection-data-stream\output...,C:\GitHub\anomaly-detection-data-stream\output...,107460,106276,1070,False,33.76504,3182.581757,0.000433,46,106230,0,0,0,2018-12-01T13:17:10.711517000Z,2018-12-01T13:34:59.733955000Z


## 6. Rótulo final das janelas

Esta célula lê somente os dois intermediários, sem reler PCAPs. Regra padrão: pelo menos um pacote inequivocamente associado a ataque torna a janela positiva. A classe vem do CSV; mais de uma classe de ataque resulta em `MULTIPLE`. Todos os pacotes associados a benigno resultam em `BENIGN`. Sem ataque confirmado e com desconhecidos, o resultado é `UNKNOWN`. Sem IPv4, `EMPTY`.

`BinaryLabel` é 1 para ataque, 0 para benigno e vazio para `UNKNOWN`/`EMPTY`. Essas linhas não devem ser convertidas automaticamente em benignas no treinamento ou na avaliação. Uma janela positiva pode conter benignos e desconhecidos: o rótulo representa a presença de ataque confirmado.

Os parâmetros abaixo permitem mudar a decisão usando as contagens salvas. A proporção usa todos os pacotes IPv4 como denominador. Se houver ataque abaixo do limiar escolhido, a janela fica `UNKNOWN`, nunca automaticamente `BENIGN`. Mudar o fuso ou a associação dos fluxos exige reconstruir o índice e repetir a extração.

**Saída esperada:** criação de `pcap_1s_labeled.csv` com metadados, 28 features, `Label` e `BinaryLabel`, seguida de uma tabela de quantidade de janelas por classe. A coluna `Label` preserva nomes como `UDP-lag` e `WebDDoS`; não força todos os ataques a um nome genérico.


In [6]:
MIN_ATTACK_PACKETS = 1
MIN_ATTACK_RATIO = 0.0

exportResult = export_labeled_dataset(
    features_path=result.featuresPath,
    counts_path=result.countsPath,
    output_path=OUTPUT_DIR / f"pcap_{WINDOW_SECONDS:g}s_labeled.csv",
    minimum_attack_packets=MIN_ATTACK_PACKETS,
    minimum_attack_ratio=MIN_ATTACK_RATIO,
)
display(pd.Series(exportResult["labelCounts"], name="Windows").rename_axis("Label").to_frame())
print("Dataset final:", exportResult["datasetPath"])


,Windows
Label,
EMPTY,113
UNKNOWN,952
BENIGN,5


Dataset final: C:\GitHub\anomaly-detection-data-stream\output\PcapStatistics\test_1million\pcap_1s_labeled.csv


## 7. Inspeção e verificação do resultado

Conferimos se cada pacote IPv4 contribuiu exatamente uma vez às features e a uma das categorias de referência. `Unknown_Packet_Count` inclui ambiguidades, ausência de correspondência e protocolos/cabeçalhos não suportados. `Attack_Class_Counts` guarda um JSON compacto com as contagens por classe e fica somente no arquivo de referência.

**Saída esperada:** primeiras linhas do dataset e das contagens, tabela de totais reconciliados e quantidade de janelas completas com rótulo binário disponível. As verificações falham explicitamente se as contagens divergirem. Nenhuma linha é removida do arquivo final nesta célula.

Cobertura elevada não prova a correção dos rótulos. Antes de processar toda a captura, confira a coerência dos horários e exemplos de comunicações no PCAP. Para uma verificação de pacotes por fluxo, escolha fluxos inteiros presentes no trecho e compare com `Total Fwd Packets`/`Total Backward Packets`; fluxos cortados pelo milhão de pacotes não devem ser comparados como completos.

Para a execução completa, após validar este teste, altere `MAX_PACKETS=None` e use outra pasta de saída, mantendo o corte e o fuso verificados. O teste não inclui os PCAPs do seu disco E: neste pacote; os resultados reais aparecerão quando você executar as células localmente.


In [7]:
dataset = pd.read_csv(exportResult["datasetPath"])
labelCounts = pd.read_csv(result.countsPath)
display(dataset.head())
display(labelCounts.head())

assert len(dataset) == len(labelCounts) == result.windowsWritten
assert dataset["Timestamp"].equals(labelCounts["Timestamp"])
assert dataset["WindowStart"].equals(labelCounts["WindowStart"])
assert (dataset["Packet_Count"] == labelCounts["Packet_Count"]).all()
assert (labelCounts["Packet_Count"] == (
    labelCounts["Benign_Packet_Count"] + labelCounts["Attack_Packet_Count"] + labelCounts["Unknown_Packet_Count"]
)).all()
assert (labelCounts["Unknown_Packet_Count"] == (
    labelCounts["Ambiguous_Packet_Count"] + labelCounts["Unmatched_Packet_Count"] + labelCounts["Unsupported_Packet_Count"]
)).all()
assert int(dataset["Packet_Count"].sum()) == result.ipv4Packets
assert list(dataset.loc[:, FEATURE_NAMES].columns) == FEATURE_NAMES
assert dataset[FEATURE_NAMES].notna().all().all()

totalColumns = ["Packet_Count", "Benign_Packet_Count", "Attack_Packet_Count", "Unknown_Packet_Count",
                "Ambiguous_Packet_Count", "Unmatched_Packet_Count", "Unsupported_Packet_Count"]
display(labelCounts[totalColumns].sum().rename("Total").to_frame())
validEvaluation = dataset["WindowComplete"].eq(1) & dataset["BinaryLabel"].isin([0, 1])
print("Janelas completas com rótulo binário disponível:", int(validEvaluation.sum()))
print("Reconciliação concluída. Nenhuma linha foi removida do dataset.")
print("Intermediários:", result.featuresPath, result.countsPath, sep="\n")


,WindowStart,Timestamp,WindowComplete,Packet_Count,IP_Bytes_Total,Packet_Length_Min,Packet_Length_Max,Packet_Length_Mean,Packet_Length_Std,IAT_Min,...,Src_Port_Unique,Dst_Port_Unique,Src_IP_Entropy,Dst_IP_Entropy,Src_Port_Entropy,Dst_Port_Entropy,TTL_Mean,TTL_Std,Label,BinaryLabel
0,2018-12-01T13:17:10.000000000Z,2018-12-01T13:17:11.000000000Z,0,0,0,0.0,0.0,0.000000,0.000000,0.0,...,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,EMPTY,NaN
1,2018-12-01T13:17:11.000000000Z,2018-12-01T13:17:12.000000000Z,1,26,7634,52.0,1125.0,293.615385,393.012143,1.0,...,3,3,0.961237,0.961237,1.460485,1.238901,63.384615,0.486504,UNKNOWN,NaN
2,2018-12-01T13:17:12.000000000Z,2018-12-01T13:17:13.000000000Z,1,2,80,40.0,40.0,40.000000,0.000000,3.0,...,1,1,-0.000000,-0.000000,-0.000000,-0.000000,241.000000,0.000000,UNKNOWN,NaN
3,2018-12-01T13:17:13.000000000Z,2018-12-01T13:17:14.000000000Z,1,10,586,40.0,80.0,58.600000,17.973314,1.0,...,3,3,1.921928,1.921928,1.584963,1.584963,98.600000,90.267602,UNKNOWN,NaN
4,2018-12-01T13:17:14.000000000Z,2018-12-01T13:17:15.000000000Z,1,12,558,41.0,52.0,46.500000,5.500000,2.0,...,5,5,1.792481,1.792481,2.251629,2.251629,123.166667,63.019618,UNKNOWN,NaN


,WindowStart,Timestamp,WindowComplete,Packet_Count,Benign_Packet_Count,Attack_Packet_Count,Unknown_Packet_Count,Ambiguous_Packet_Count,Unmatched_Packet_Count,Unsupported_Packet_Count,Attack_Packet_Ratio,Label_Coverage,Attack_Class_Counts
0,2018-12-01T13:17:10.000000000Z,2018-12-01T13:17:11.000000000Z,0,0,0,0,0,0,0,0,0.0,0.0,{}
1,2018-12-01T13:17:11.000000000Z,2018-12-01T13:17:12.000000000Z,1,26,0,0,26,0,26,0,0.0,0.0,{}
2,2018-12-01T13:17:12.000000000Z,2018-12-01T13:17:13.000000000Z,1,2,0,0,2,0,2,0,0.0,0.0,{}
3,2018-12-01T13:17:13.000000000Z,2018-12-01T13:17:14.000000000Z,1,10,0,0,10,0,6,4,0.0,0.0,{}
4,2018-12-01T13:17:14.000000000Z,2018-12-01T13:17:15.000000000Z,1,12,0,0,12,0,12,0,0.0,0.0,{}


,Total
Packet_Count,106276
Benign_Packet_Count,46
Attack_Packet_Count,0
Unknown_Packet_Count,106230
Ambiguous_Packet_Count,0
Unmatched_Packet_Count,105391
Unsupported_Packet_Count,839


Janelas completas com rótulo binário disponível: 5
Reconciliação concluída. Nenhuma linha foi removida do dataset.
Intermediários:
C:\GitHub\anomaly-detection-data-stream\output\PcapStatistics\test_1million\window_features.csv
C:\GitHub\anomaly-detection-data-stream\output\PcapStatistics\test_1million\window_label_counts.csv
